# Version 2.0 Candidate Hillslope Control Extraction Pipeline (Colab GEE)
## Himalayan Landslide Intelligence — Phase 3b Multi-Sensor Extraction (Step 5.3)

### Scientific Context & Hydrological Governance:
1. **Terminology & Failure Status:** Negative controls are designated **Candidate Hillslope Control / Unrecorded Failure Status**. Absence from the NASA Global Landslide Catalog establishes unrecorded failure status, not guaranteed absolute stability across geological timescales.
2. **Confound Mitigation Policy:** Sampling within the empirical hillslope envelope ($6.1^\circ \le \text{slope} \le 41.4^\circ$, $266\text{ m} \le \text{elev} \le 3027\text{ m}$) is designed to mitigate the historical flat-terrain bias (Version 1.0 controls had mean slope $1.43^\circ$). **Actual elimination of the terrain confound cannot be assumed a priori**; it must be empirically validated via two-sample Kolmogorov-Smirnov tests on the extracted feature distributions.
3. **SAR Temporal Consistency Screening:** Pre-event Sentinel-1 IW GRD VV standard deviation must be $\le 1.5\text{ dB}$ over a 180-day baseline observation window ($[t_{\text{event}}-180\text{d}, t_{\text{event}}-15\text{d}]$), with at least 3 valid acquisitions required. *Note:* This serves as a microwave temporal consistency screening criterion, not geodetic proof of ground stability; backscatter variance may also be influenced by seasonal freeze-thaw, vegetation phenology, episodic soil moisture spikes, and radar layover/shadow on steep terrain. Potential selection asymmetry between controls and external positive inventories is evaluated through sensitivity analysis.
4. **Rigorous Data Availability Masks (No Silent Defaults):**
   - JRC Surface Water ($< 10\%$) and ESA WorldCover (Class 80 $\neq$ Water, Class 70 $\neq$ Snow/Ice) are evaluated with strict valid-data masks. Missing water or land-cover observations are explicitly rejected, NOT converted to zero.
   - Mandatory terrain variables (elevation, slope, aspect, curvature, TRI, TWI) are subject to explicit null checks on both controls and positive events; no arbitrary fallbacks (e.g. slope=21.0, elev=1200m) are permitted.
5. **Native Hydrological Grid Resolution for TWI:**
   - MERIT Hydro 90m native grid (3 arc-seconds) is preserved for upstream drainage area (`upa`).
   - Specific catchment area is calculated using native 90m contour width: $\alpha = (\text{upa} \times 10^6) / 90.0\text{ m}$.
   - SRTM 30m slope is spatially aggregated (mean) to the 90m hydrological grid to avoid fictitious 30m hydrological resampling artifacts.
   - $\text{TWI} = \ln(\alpha / \tan\beta)$, with $\tan\beta \ge 0.001$. Slope-only proxy is preserved separately.
6. **Multi-Batch Candidate Optimization:** All 3 candidate batches are evaluated sequentially. Candidates are deduplicated by coordinates (30m tolerance). The representative control is selected by minimizing the standardized three-feature topographic distance $D_{\text{topo}}$ across the pooled candidate set. The result is transparently documented as the minimum-distance candidate among sampled valid candidates.
7. **Audit & Diagnostics:** Complete rejection audit tracking 13 categories, cross-event duplicate control audit, and full percentile distributions of matching differences.

In [ ]:
# Step 1: Install Dependencies and Initialize Google Earth Engine
!pip install -q geemap earthengine-api pyproj pandas numpy scipy

import ee
import geemap.core as geemap
import pandas as pd
import numpy as np
import time
import json
import math
from datetime import datetime

# Authenticate and Initialize GEE with Project Credentials
try:
    ee.Initialize(project='landslidedetection-506008')
    print('Earth Engine already initialized from existing credentials.')
except Exception:
    try:
        from google.colab import auth
        auth.authenticate_user()
        ee.Initialize(project='landslidedetection-506008')
        print('Earth Engine initialized via Colab user authentication.')
    except Exception:
        # Fallback to manual notebook auth mode (avoids credential propagation error)
        ee.Authenticate(auth_mode='notebook')
        ee.Initialize(project='landslidedetection-506008')
        print('Earth Engine initialized via notebook authorization code.')

print('Earth Engine initialized successfully:', ee.Number(1).add(1).getInfo() == 2)

In [ ]:
# Step 2: Load Confirmed Positive Landslide Events
url_p2 = 'https://raw.githubusercontent.com/GouravSharma26/LandSlide/main/outputs/phase2_cleaned_dataset.csv'
df_p2 = pd.read_csv(url_p2)
df_p2['event_date'] = pd.to_datetime(df_p2['event_date'])
positives = df_p2[df_p2['label'] == 1].reset_index(drop=True)
print(f'Loaded {len(positives)} confirmed positive rainfall-triggered landslide events.')

# Authoritative Himalayan Region of Interest (ROI)
lat_min, lat_max = 26.0, 35.0
lon_min, lon_max = 74.0, 97.0
roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])

# Universal Exclusion Zone: 2000m geodesic buffer around ALL 127 positive events
pos_points = [ee.Geometry.Point([row['longitude'], row['latitude']]) for _, row in positives.iterrows()]
universal_exclusion = ee.FeatureCollection([ee.Feature(pt.buffer(2000)) for pt in pos_points]).geometry()
all_positives_coords = [(float(r['latitude']), float(r['longitude'])) for _, r in positives.iterrows()]
print('Authoritative Himalayan ROI and Universal Exclusion Zone (2.0 km) declared successfully.')

In [ ]:
# Step 3: Environmental Datasets & Geomorphometric Indices (Riley TRI & Native 90m MERIT TWI)

# 1. SRTM 30m Digital Elevation Model & Derivatives
srtm = ee.Image('USGS/SRTMGL1_003').select('elevation').clip(roi)
slope_30m = ee.Terrain.slope(srtm)
aspect_30m = ee.Terrain.aspect(srtm)
curvature_30m = srtm.convolve(ee.Kernel.laplacian8(normalize=False)).rename('curvature')

# 2. Corrected Riley et al. (1999) Terrain Ruggedness Index (TRI)
# Formula: sqrt( sum_{i=1..8} (z_i - z_0)^2 )
offsets = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
sq_diff_sum = ee.Image(0)
abs_diff_sum = ee.Image(0)
for dx, dy in offsets:
    weights = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
    weights[dy + 1][dx + 1] = 1
    k = ee.Kernel.fixed(3, 3, weights)
    neighbor = srtm.convolve(k)
    diff = neighbor.subtract(srtm)
    sq_diff_sum = sq_diff_sum.add(diff.pow(2))
    abs_diff_sum = abs_diff_sum.add(diff.abs())

tri_riley = sq_diff_sum.sqrt().rename('tri')
tri_mean_diff = abs_diff_sum.divide(8.0).rename('tri_mean_diff')
tri_focal_proxy = srtm.subtract(srtm.convolve(ee.Kernel.square(radius=1, units='pixels'))).abs().rename('tri_focal_diff_proxy')

# 3. Topographic Wetness Index (TWI) on Native MERIT Hydro 90m Hydrological Grid
# Native 3 arc-second (~90m) hydrological routing is preserved without fictitious 30m resampling.
# SRTM 30m slope is spatially aggregated (mean) to the 90m hydrological cell to represent cell gradient.
merit_hydro = ee.Image('MERIT/Hydro/v1_0_1').clip(roi)
upa_90m = merit_hydro.select('upa')  # Upstream area in km^2
alpha_90m = upa_90m.multiply(1e6).divide(90.0).max(1.0)  # Specific catchment area with contour width b=90.0m

# Spatially aggregate SRTM 30m slope to MERIT Hydro 90m projection
slope_90m = slope_30m.reduceResolution(reducer=ee.Reducer.mean(), maxPixels=16).reproject(crs=merit_hydro.projection())
slp_rad_90m = slope_90m.multiply(math.pi / 180.0)
tan_beta_90m = slp_rad_90m.tan().max(0.001)
twi_merit_90m = alpha_90m.divide(tan_beta_90m).log().rename('twi')
twi_slope_proxy_90m = tan_beta_90m.pow(-1).log().rename('twi_slope_proxy')

# 4. Surface Water Occurrence: JRC Global Surface Water (v1.4) — PRESERVED WITHOUT unmask(0)
jrc_water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(roi)

# 5. Land Cover & Permanent Snow/Ice: ESA WorldCover 10m (v100)
# Class 70 = Snow and ice; Class 80 = Permanent water bodies; Class 50 = Built-up
worldcover = ee.ImageCollection('ESA/WorldCover/v100').first().select('Map').clip(roi)

# Multi-band Static Diagnostic Image for Rapid Batch Candidate Evaluation
static_diag = (
    srtm.rename('elevation')
    .addBands(slope_30m.rename('slope'))
    .addBands(aspect_30m.rename('aspect'))
    .addBands(curvature_30m.rename('curvature'))
    .addBands(tri_riley)
    .addBands(tri_mean_diff)
    .addBands(tri_focal_proxy)
    .addBands(twi_merit_90m)
    .addBands(twi_slope_proxy_90m)
    .addBands(jrc_water.rename('water_occurrence'))  # Valid-data mask preserved
    .addBands(worldcover.rename('landcover'))
)
print('Static environmental layers, Riley TRI, and Native 90m MERIT TWI compiled successfully.')

In [ ]:
# Step 4: Strict Positive Validation & Multi-Batch Candidate Optimization

def haversine_km(lat1, lon1, lat2, lon2):
    """Calculates geodesic distance in km between two lat/lon points."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.asin(math.sqrt(a))

# Standard deviations of confirmed positive events for standardized three-feature distance
SIGMA_SLOPE = 10.5
SIGMA_ELEV = 775.0
SIGMA_TRI = 17.5

# Comprehensive Rejection Audit Tracking (13 Categories)
sampling_audit = {
    'total_positive_events_in_catalog': len(positives),
    'positive_events_validated': 0,
    'positive_events_excluded_from_matching': 0,
    'total_candidates_evaluated': 0,
    'controls_successfully_sampled': 0,
    'rejections': {
        'roi_out_of_bounds': 0,
        'scar_proximity_under_2km': 0,
        'missing_terrain_data': 0,
        'missing_water_data': 0,
        'missing_landcover_data': 0,
        'elevation_out_of_bounds': 0,
        'slope_out_of_bounds': 0,
        'water_body_overlap': 0,
        'snow_ice_overlap': 0,
        'sar_insufficient_baseline': 0,
        'sar_temporal_instability': 0,
        'duplicate_candidate_coordinates': 0,
        'no_valid_candidate_after_search': 0
    }
}

v2_controls = []
matching_diagnostics = []
positive_events_excluded_audit = []

print('Starting Candidate Hillslope Control Sampling across confirmed positive events...')
print('Screening: Proximity > 2km | Elev [266-3027m] | Slope [6.1-41.4 deg] | Water < 10% | Snow/Ice != 70 | Pre-SAR std <= 1.5 dB')
start_sampling = time.time()

for idx, row in positives.iterrows():
    p_lat, p_lon = float(row['latitude']), float(row['longitude'])
    event_date = pd.to_datetime(row['event_date'])
    pt_pos = ee.Geometry.Point([p_lon, p_lat])
    
    # 1. STRICT POSITIVE-EVENT TERRAIN VALIDATION (NO ARBITRARY DEFAULTS)
    pos_terrain = static_diag.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=pt_pos,
        scale=30,
        bestEffort=True
    ).getInfo() or {}
    
    p_slope = pos_terrain.get('slope')
    p_elev = pos_terrain.get('elevation')
    p_asp = pos_terrain.get('aspect')
    p_tri = pos_terrain.get('tri')
    p_curv = pos_terrain.get('curvature')
    p_twi = pos_terrain.get('twi')
    
    # Check for missing terrain in positive event
    pos_missing = []
    for var_name, val in [('slope', p_slope), ('elevation', p_elev), ('aspect', p_asp), ('tri', p_tri)]:
        if val is None:
            pos_missing.append(var_name)
            
    if pos_missing or not (lat_min <= p_lat <= lat_max and lon_min <= p_lon <= lon_max):
        sampling_audit['positive_events_excluded_from_matching'] += 1
        positive_events_excluded_audit.append({
            'paired_positive_idx': int(idx),
            'latitude': p_lat,
            'longitude': p_lon,
            'event_date': event_date.strftime('%Y-%m-%d'),
            'exclusion_reason': 'missing_positive_terrain_values',
            'missing_variables': pos_missing
        })
        print(f'Warning: Positive event {idx} excluded from matching (Missing: {pos_missing})')
        continue
        
    sampling_audit['positive_events_validated'] += 1
    p_slope, p_elev, p_asp, p_tri = float(p_slope), float(p_elev), float(p_asp), float(p_tri)
    
    # 2. LOCAL SEARCH RING & PRE-EVENT SAR STACK
    # 10km to 40km local catchment ring buffer, intersected with regional ROI
    search_ring = pt_pos.buffer(40000).difference(pt_pos.buffer(10000)).intersection(roi)
    
    # Pre-event Sentinel-1 baseline window: [event_date - 180 days, event_date - 15 days]
    pre_start = event_date - pd.Timedelta(days=180)
    pre_end = event_date - pd.Timedelta(days=15)
    s1_pre_coll = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filterBounds(search_ring)
        .filterDate(pre_start.strftime('%Y-%m-%d'), pre_end.strftime('%Y-%m-%d'))
    )
    s1_std_vv = s1_pre_coll.select('VV').reduce(ee.Reducer.stdDev()).rename('sar_pre_std_vv')
    s1_count = s1_pre_coll.select('VV').reduce(ee.Reducer.count()).rename('sar_pre_count')
    
    event_diag = static_diag.addBands(s1_std_vv).addBands(s1_count)
    
    # 3. MULTI-BATCH ACCUMULATION & DEDUPLICATION (DO NOT BREAK EARLY)
    pooled_valid_candidates = []
    seen_batch_coords = []
    num_batches = 3
    
    for batch_num in range(num_batches):
        batch_pts = ee.FeatureCollection.randomPoints(
            region=search_ring,
            points=25,
            seed=42 + idx * 100 + batch_num * 31
        )
        candidates = event_diag.sampleRegions(
            collection=batch_pts,
            scale=30,
            geometries=True
        ).getInfo().get('features', [])
        
        for cand in candidates:
            sampling_audit['total_candidates_evaluated'] += 1
            coords = cand['geometry']['coordinates']
            c_lon, c_lat = float(coords[0]), float(coords[1])
            props = cand.get('properties', {})
            
            # Filter 1: Regional ROI bounds
            if not (lat_min <= c_lat <= lat_max and lon_min <= c_lon <= lon_max):
                sampling_audit['rejections']['roi_out_of_bounds'] += 1
                continue
                
            # Filter 2: Proximity to ANY known positive event (> 2.0 km)
            min_dist_to_positive = min(
                haversine_km(c_lat, c_lon, p_coord[0], p_coord[1])
                for p_coord in all_positives_coords
            )
            if min_dist_to_positive <= 2.0:
                sampling_audit['rejections']['scar_proximity_under_2km'] += 1
                continue
                
            # Filter 3: Explicit check for mandatory terrain features
            c_elev = props.get('elevation')
            c_slope = props.get('slope')
            c_aspect = props.get('aspect')
            c_curv = props.get('curvature')
            c_tri = props.get('tri')
            c_tri_mean = props.get('tri_mean_diff')
            c_tri_proxy = props.get('tri_focal_diff_proxy')
            c_twi = props.get('twi')
            c_twi_proxy = props.get('twi_slope_proxy')
            
            if any(v is None for v in [c_elev, c_slope, c_aspect, c_curv, c_tri, c_tri_mean, c_tri_proxy, c_twi, c_twi_proxy]):
                sampling_audit['rejections']['missing_terrain_data'] += 1
                continue
                
            # Filter 4: Water observation availability
            c_water = props.get('water_occurrence')
            if c_water is None:
                sampling_audit['rejections']['missing_water_data'] += 1
                continue
                
            # Filter 5: Land-cover observation availability
            c_lc = props.get('landcover')
            if c_lc is None:
                sampling_audit['rejections']['missing_landcover_data'] += 1
                continue
                
            # Filter 6: Elevation envelope [266m, 3027m]
            if float(c_elev) < 266.0 or float(c_elev) > 3027.0:
                sampling_audit['rejections']['elevation_out_of_bounds'] += 1
                continue
                
            # Filter 7: Slope envelope [6.1 deg, 41.4 deg]
            if float(c_slope) < 6.1 or float(c_slope) > 41.4:
                sampling_audit['rejections']['slope_out_of_bounds'] += 1
                continue
                
            # Filter 8: Water body overlap (JRC >= 10% or WorldCover == 80)
            if float(c_water) >= 10.0 or int(c_lc) == 80:
                sampling_audit['rejections']['water_body_overlap'] += 1
                continue
                
            # Filter 9: Permanent snow/ice overlap (WorldCover == 70)
            if int(c_lc) == 70:
                sampling_audit['rejections']['snow_ice_overlap'] += 1
                continue
                
            # Filter 10: SAR temporal baseline count (>= 3 acquisitions)
            c_sar_count = props.get('sar_pre_count')
            c_sar_std = props.get('sar_pre_std_vv')
            if c_sar_count is None or int(c_sar_count) < 3 or c_sar_std is None:
                sampling_audit['rejections']['sar_insufficient_baseline'] += 1
                continue
                
            # Filter 11: SAR temporal consistency screening criterion (Std <= 1.5 dB)
            if float(c_sar_std) > 1.5:
                sampling_audit['rejections']['sar_temporal_instability'] += 1
                continue
                
            # Coordinate Deduplication Check (30m tolerance)
            is_duplicate = False
            for prev_lat, prev_lon in seen_batch_coords:
                if haversine_km(c_lat, c_lon, prev_lat, prev_lon) <= 0.03:
                    is_duplicate = True
                    break
            if is_duplicate:
                sampling_audit['rejections']['duplicate_candidate_coordinates'] += 1
                continue
                
            seen_batch_coords.append((c_lat, c_lon))
            
            # Standardized three-feature topographic distance
            d_slope = abs(float(c_slope) - p_slope)
            d_elev = abs(float(c_elev) - p_elev)
            d_tri = abs(float(c_tri) - p_tri)
            d_aspect = min(abs(float(c_aspect) - p_asp), 360.0 - abs(float(c_aspect) - p_asp))
            d_curv = abs(float(c_curv) - float(p_curv)) if p_curv is not None else None
            d_twi = abs(float(c_twi) - float(p_twi)) if p_twi is not None else None
            dist_paired = haversine_km(p_lat, p_lon, c_lat, c_lon)
            
            d_topo_3feat = math.sqrt(
                (d_slope / SIGMA_SLOPE) ** 2 +
                (d_elev / SIGMA_ELEV) ** 2 +
                (d_tri / SIGMA_TRI) ** 2
            )
            
            cand_record = {
                'latitude': c_lat,
                'longitude': c_lon,
                'event_date': event_date.strftime('%Y-%m-%d'),
                'Location Accuracy': 'Candidate Hillslope Control / Unrecorded Failure Status',
                'Landslide Trigger': 'Candidate Hillslope Control / Unrecorded Failure Status',
                'Imported Event Source Catalog': 'V2_GEE_Candidate_Hillslope',
                'label': 0,
                'block_id': f"{int(c_lon//0.5)}_{int(c_lat//0.5)}",
                'paired_positive_idx': int(idx),
                'sar_pre_std_vv': float(c_sar_std),
                'sar_pre_count': int(c_sar_count),
                'elevation': float(c_elev),
                'slope': float(c_slope),
                'aspect': float(c_aspect),
                'curvature': float(c_curv),
                'tri': float(c_tri),
                'tri_mean_diff': float(c_tri_mean),
                'tri_focal_diff_proxy': float(c_tri_proxy),
                'twi': float(c_twi),
                'twi_slope_proxy': float(c_twi_proxy),
                'min_distance_to_any_positive_km': float(min_dist_to_positive),
                'spatial_distance_to_paired_positive_km': float(dist_paired),
                'delta_slope_deg': float(d_slope),
                'delta_elevation_m': float(d_elev),
                'delta_aspect_deg': float(d_aspect),
                'delta_tri': float(d_tri),
                'delta_curvature': float(d_curv) if d_curv is not None else None,
                'delta_twi': float(d_twi) if d_twi is not None else None,
                'three_feature_topographic_distance': float(d_topo_3feat),
                'd_topo_composite': float(d_topo_3feat),
                'discovered_batch_idx': batch_num
            }
            pooled_valid_candidates.append(cand_record)
            
    # 4. SELECT MINIMUM-DISTANCE CANDIDATE AFTER ALL 3 BATCHES FINISH
    if pooled_valid_candidates:
        pooled_valid_candidates.sort(key=lambda r: r['three_feature_topographic_distance'])
        best_candidate = pooled_valid_candidates[0]
        v2_controls.append(best_candidate)
        sampling_audit['controls_successfully_sampled'] += 1
        
        matching_diagnostics.append({
            'paired_positive_idx': int(idx),
            'control_lat': best_candidate['latitude'],
            'control_lon': best_candidate['longitude'],
            'positive_lat': p_lat,
            'positive_lon': p_lon,
            'spatial_distance_km': best_candidate['spatial_distance_to_paired_positive_km'],
            'min_dist_to_all_positives_km': best_candidate['min_distance_to_any_positive_km'],
            'control_slope': best_candidate['slope'],
            'positive_slope': p_slope,
            'delta_slope_deg': best_candidate['delta_slope_deg'],
            'control_elevation': best_candidate['elevation'],
            'positive_elevation': p_elev,
            'delta_elevation_m': best_candidate['delta_elevation_m'],
            'control_aspect': best_candidate['aspect'],
            'positive_aspect': p_asp,
            'delta_aspect_deg': best_candidate['delta_aspect_deg'],
            'control_tri': best_candidate['tri'],
            'positive_tri': p_tri,
            'delta_tri': best_candidate['delta_tri'],
            'delta_curvature': best_candidate.get('delta_curvature'),
            'delta_twi': best_candidate.get('delta_twi'),
            'three_feature_topographic_distance': best_candidate['three_feature_topographic_distance'],
            'sar_pre_std_vv': best_candidate['sar_pre_std_vv'],
            'sar_pre_count': best_candidate['sar_pre_count'],
            'selected_from_batch_idx': best_candidate['discovered_batch_idx'],
            'total_valid_candidates_evaluated': len(pooled_valid_candidates),
            'selection_characterization': 'Minimum standardized distance among sampled valid candidates'
        })
    else:
        sampling_audit['rejections']['no_valid_candidate_after_search'] += 1
        print(f'Warning: No valid candidate found for positive event {idx} across all {num_batches} batches.')
        
    if (idx + 1) % 25 == 0 or (idx + 1) == len(positives):
        print(f'  Progress: {len(v2_controls)} controls accepted / {idx+1} events evaluated ({sampling_audit["total_candidates_evaluated"]} candidates screened)')

df_v2_controls = pd.DataFrame(v2_controls)
df_matching_diagnostics = pd.DataFrame(matching_diagnostics)
df_positive_exclusions = pd.DataFrame(positive_events_excluded_audit)
elapsed_sampling = time.time() - start_sampling
print(f'\nSampling Complete in {elapsed_sampling:.1f}s.')
print(f'Controls Sampled: {len(df_v2_controls)} | Positive Events Excluded: {len(df_positive_exclusions)}')
print('Detailed Rejection Tallies across all 13 Categories:')
for reason, count in sampling_audit['rejections'].items():
    print(f'  - {reason}: {count}')

df_v2_controls.to_csv('phase2_v2_hillslope_controls.csv', index=False)
df_matching_diagnostics.to_csv('v2_control_matching_diagnostics.csv', index=False)
df_positive_exclusions.to_csv('v2_positive_events_excluded_audit.csv', index=False)

In [ ]:
# Step 5: Strict Validation, Duplicate Control Audit & Full Distribution Reporting
print('Executing Candidate Control Constraint Validation...')
assert len(df_v2_controls) > 0, 'No candidate controls sampled!'
assert df_v2_controls['latitude'].between(lat_min, lat_max).all(), 'Latitude outside ROI bounds!'
assert df_v2_controls['longitude'].between(lon_min, lon_max).all(), 'Longitude outside ROI bounds!'
assert (df_v2_controls['min_distance_to_any_positive_km'] > 2.0).all(), 'Universal exclusion violation (point <= 2km from scar)!'
assert df_v2_controls['slope'].between(6.1, 41.4).all(), 'Slope outside empirical envelope [6.1, 41.4]!'
assert df_v2_controls['elevation'].between(266, 3027).all(), 'Elevation outside empirical envelope [266, 3027]!'
assert (df_v2_controls['sar_pre_std_vv'] <= 1.5).all(), 'SAR consistency threshold violated (> 1.5 dB)!'
assert df_v2_controls[['elevation', 'slope', 'aspect', 'curvature', 'tri', 'twi']].notna().all().all(), 'Null terrain values detected!'
print('PASSED: All candidate controls satisfy geographic bounds, universal exclusion (>2km), terrain envelope, and SAR consistency.')

# Cross-Event Duplicate Control & Minimum Pairwise Separation Audit
n_total_ctrls = len(df_v2_controls)
n_unique_ctrls = len(df_v2_controls[['latitude', 'longitude']].drop_duplicates())
duplicate_controls_cross_event = n_total_ctrls - n_unique_ctrls

# Calculate pairwise minimum separation distance between selected controls
ctrl_coords = list(zip(df_v2_controls['latitude'], df_v2_controls['longitude']))
min_pairwise_ctrl_dist_km = 999999.0
for i in range(len(ctrl_coords)):
    for j in range(i + 1, len(ctrl_coords)):
        d = haversine_km(ctrl_coords[i][0], ctrl_coords[i][1], ctrl_coords[j][0], ctrl_coords[j][1])
        if d < min_pairwise_ctrl_dist_km:
            min_pairwise_ctrl_dist_km = d

print(f'\nCross-Event Duplicate Controls: {duplicate_controls_cross_event} / {n_total_ctrls}')
print(f'Minimum Pairwise Control Separation: {min_pairwise_ctrl_dist_km:.2f} km')

# Full Empirical Distributions of Matching Differences
print('\nFull Empirical Distribution of Topographic Matching Differences:')
dist_metrics = [
    'spatial_distance_km',
    'delta_slope_deg',
    'delta_elevation_m',
    'delta_aspect_deg',
    'delta_tri',
    'three_feature_topographic_distance'
]
summary_rows = []
for m in dist_metrics:
    s = df_matching_diagnostics[m].dropna()
    summary_rows.append({
        'Metric': m,
        'Min': round(float(s.min()), 2),
        'P25': round(float(s.quantile(0.25)), 2),
        'Median': round(float(s.median()), 2),
        'Mean': round(float(s.mean()), 2),
        'P75': round(float(s.quantile(0.75)), 2),
        'Max': round(float(s.max()), 2)
    })
df_dist_summary = pd.DataFrame(summary_rows)
display(df_dist_summary)
df_dist_summary.to_csv('v2_matching_distribution_summary.csv', index=False)

In [ ]:
# Step 6: Declare Dynamic Multi-Sensor Collections

# 1. Sentinel-1 IW GRD C-Band SAR
s1_grd = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filterBounds(roi)
)

# 2. Sentinel-2 MSI Surface Reflectance + Cloudless Probability Join
s2_sr = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 70))
)
s2_clouds = ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY').filterBounds(roi)
s2_with_prob = ee.Join.saveFirst('cloud_prob').apply(
    primary=s2_sr,
    secondary=s2_clouds,
    condition=ee.Filter.equals(leftField='system:index', rightField='system:index')
)

# Robust S2 Cloud Masking: Combines Scene Classification Layer (SCL) and Cloud Probability
def mask_s2_clouds(img):
    scl = img.select('SCL')
    scl_valid = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    c_prob_obj = img.get('cloud_prob')
    c_prob_mask = ee.Algorithms.If(
        c_prob_obj,
        ee.Image(c_prob_obj).select('probability').lt(40),
        ee.Image(1)
    )
    return img.updateMask(scl_valid.And(ee.Image(c_prob_mask)))

# 3. CHIRPS Daily Precipitation (~5.5 km resolution)
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterBounds(roi)
print('Dynamic satellite collections and robust masking functions declared successfully.')

In [ ]:
# Step 7: Robust Extraction Functions (Multi-Scale SAR, CHIRPS, Adaptive Optical)

SAR_NOMINAL_WINDOW_DAYS = 12
SAR_EXTENDED_WINDOW_DAYS = 24
SPECTRAL_WINDOW_STEPS = [30, 60, 90, 120]
API_DECAY = 0.90

def extract_point_and_buffers_sar(lon, lat, event_date_str):
    """Extracts Sentinel-1 SAR at 10m point, 500m buffer, and 1000m buffer with adaptive window fallback."""
    pt = ee.Geometry.Point([lon, lat])
    event = ee.Date(event_date_str)
    
    pre_coll = s1_grd.filterBounds(pt).filterDate(event.advance(-SAR_NOMINAL_WINDOW_DAYS, 'day'), event)
    post_coll = s1_grd.filterBounds(pt).filterDate(event, event.advance(SAR_NOMINAL_WINDOW_DAYS, 'day'))
    
    window_used = SAR_NOMINAL_WINDOW_DAYS
    pre_size = pre_coll.size().getInfo()
    post_size = post_coll.size().getInfo()
    
    if pre_size == 0 or post_size == 0:
        pre_coll = s1_grd.filterBounds(pt).filterDate(event.advance(-SAR_EXTENDED_WINDOW_DAYS, 'day'), event)
        post_coll = s1_grd.filterBounds(pt).filterDate(event, event.advance(SAR_EXTENDED_WINDOW_DAYS, 'day'))
        window_used = SAR_EXTENDED_WINDOW_DAYS
        pre_size = pre_coll.size().getInfo()
        post_size = post_coll.size().getInfo()
        
    if pre_size == 0 or post_size == 0:
        return {
            'sar_window_days': None,
            'sar_quality_flag': 'NO_SAR_PASS_IN_WINDOW',
            'VV_pre': None, 'VH_pre': None, 'VV_post': None, 'VH_post': None, 'dVV': None, 'dVH': None,
            'VV_pre_500m': None, 'VH_pre_500m': None, 'VV_post_500m': None, 'VH_post_500m': None,
            'dVV_500m': None, 'dVH_500m': None, 'VV_pre_1000m': None, 'VH_pre_1000m': None,
            'VV_post_1000m': None, 'VH_post_1000m': None, 'dVV_1000m': None, 'dVH_1000m': None
        }
        
    pre = pre_coll.mean()
    post = post_coll.mean()
    delta = post.subtract(pre)
    
    img = (
        pre.select(['VV', 'VH'], ['VV_pre', 'VH_pre'])
        .addBands(post.select(['VV', 'VH'], ['VV_post', 'VH_post']))
        .addBands(delta.select(['VV', 'VH'], ['dVV', 'dVH']))
    )
    
    # 1. 10m point sample
    pt_res = img.reduceRegion(reducer=ee.Reducer.first(), geometry=pt, scale=10, bestEffort=True).getInfo() or {}
    # 2. 500m circular buffer mean and minimum
    res_500m = img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.min(), '_min', True),
        geometry=pt.buffer(500),
        scale=10,
        bestEffort=True
    ).getInfo() or {}
    # 3. 1000m circular buffer mean and minimum
    res_1000m = img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.min(), '_min', True),
        geometry=pt.buffer(1000),
        scale=20,
        bestEffort=True
    ).getInfo() or {}
    
    combined = {'sar_window_days': window_used, 'sar_quality_flag': 'VALID'}
    combined.update(pt_res)
    for k, v in res_500m.items():
        combined[f'{k}_500m'] = v
    for k, v in res_1000m.items():
        combined[f'{k}_1000m'] = v
    return combined

def extract_rainfall(lon, lat, event_date_str):
    """Extracts CHIRPS precipitation: 1d, 7d, 30d, and API (decay=0.90)."""
    pt = ee.Geometry.Point([lon, lat])
    event = ee.Date(event_date_str)
    
    r1 = chirps.filterDate(event.advance(-1, 'day'), event).sum().rename('rain_1d')
    r7 = chirps.filterDate(event.advance(-7, 'day'), event).sum().rename('rain_7d')
    r30 = chirps.filterDate(event.advance(-30, 'day'), event).sum().rename('rain_30d')
    
    api_window = chirps.filterDate(event.advance(-30, 'day'), event)
    def weight(img):
        days = event.difference(img.date(), 'day')
        return img.multiply(ee.Number(API_DECAY).pow(days))
    api = api_window.map(weight).sum().rename('API')
    
    img = r1.addBands(r7).addBands(r30).addBands(api)
    res = img.reduceRegion(reducer=ee.Reducer.first(), geometry=pt, scale=5566, bestEffort=True).getInfo() or {}
    return res

def extract_spectral(lon, lat, event_date_str):
    """Adaptive multi-window Sentinel-2 spectral extraction for NDVI, NDWI, and BSI."""
    pt = ee.Geometry.Point([lon, lat])
    end = ee.Date(event_date_str)
    
    for window in SPECTRAL_WINDOW_STEPS:
        start = end.advance(-window, 'day')
        coll = s2_with_prob.filterBounds(pt).filterDate(start, end).map(mask_s2_clouds)
        coll_size = coll.size().getInfo()
        if coll_size == 0:
            continue
            
        comp = coll.median()
        ndvi = comp.normalizedDifference(['B8', 'B4']).rename('NDVI')
        ndwi = comp.normalizedDifference(['B3', 'B8']).rename('NDWI')
        bsi = comp.expression(
            '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
            {'SWIR': comp.select('B11'), 'RED': comp.select('B4'),
             'NIR': comp.select('B8'), 'BLUE': comp.select('B2')}
        ).rename('BSI')
        
        img = ndvi.addBands(ndwi).addBands(bsi)
        res = img.reduceRegion(reducer=ee.Reducer.first(), geometry=pt, scale=10, bestEffort=True).getInfo()
        if res and res.get('NDVI') is not None:
            res['spectral_window_days'] = window
            res['optical_quality_flag'] = 'VALID'
            return res
            
    return {
        'NDVI': None, 'NDWI': None, 'BSI': None,
        'spectral_window_days': None,
        'optical_quality_flag': 'PERSISTENT_MONSOON_CLOUD_COVER'
    }

print('Extraction functions compiled successfully with robust null-safety and fallbacks.')

In [ ]:
# Step 8: Sequential Feature Extraction Loop across all Candidate Controls
extracted_controls = []
n_total = len(df_v2_controls)
print(f'Extracting multisource dynamic features for {n_total} candidate controls...')
start_loop = time.time()

quality_audit = {
    'sar_complete': 0,
    'sar_extended_window': 0,
    'sar_data_gap': 0,
    'rain_complete': 0,
    'optical_valid': 0,
    'optical_cloud_covered': 0
}

for i, row in df_v2_controls.iterrows():
    rec = row.to_dict()
    lon, lat, date_str = row['longitude'], row['latitude'], row['event_date']
    
    # 1. SAR Point & Buffers
    try:
        sar_dict = extract_point_and_buffers_sar(lon, lat, date_str)
        rec.update(sar_dict)
        if sar_dict.get('sar_quality_flag') == 'VALID':
            quality_audit['sar_complete'] += 1
            if sar_dict.get('sar_window_days') == SAR_EXTENDED_WINDOW_DAYS:
                quality_audit['sar_extended_window'] += 1
        else:
            quality_audit['sar_data_gap'] += 1
    except Exception as e:
        rec['sar_quality_flag'] = f'ERROR: {str(e)[:50]}'
        quality_audit['sar_data_gap'] += 1
        
    # 2. Rainfall
    try:
        rain_dict = extract_rainfall(lon, lat, date_str)
        rec.update(rain_dict)
        if rain_dict.get('rain_1d') is not None:
            quality_audit['rain_complete'] += 1
    except Exception as e:
        rec['_rain_error'] = str(e)[:50]
        
    # 3. Spectral Indices
    try:
        spec_dict = extract_spectral(lon, lat, date_str)
        rec.update(spec_dict)
        if spec_dict.get('optical_quality_flag') == 'VALID':
            quality_audit['optical_valid'] += 1
        else:
            quality_audit['optical_cloud_covered'] += 1
    except Exception as e:
        rec['optical_quality_flag'] = f'ERROR: {str(e)[:50]}'
        quality_audit['optical_cloud_covered'] += 1
        
    extracted_controls.append(rec)
    if (i + 1) % 15 == 0 or (i + 1) == n_total:
        elapsed = time.time() - start_loop
        print(f'  Progress: {i+1}/{n_total} controls extracted ({elapsed:.1f}s elapsed)')

df_v2_extracted_controls = pd.DataFrame(extracted_controls)
print('\nDynamic Feature Extraction Complete.')
print('Extraction Quality Audit:', quality_audit)

In [ ]:
# Step 9: Assemble Version 2.0 Feature Table & Export Artifacts
url_p3 = 'https://raw.githubusercontent.com/GouravSharma26/LandSlide/main/outputs/phase3_feature_table.csv'
df_p3 = pd.read_csv(url_p3)
df_p3_positives = df_p3[df_p3['label'] == 1].reset_index(drop=True)

# Common schema alignment
common_cols = [c for c in df_p3_positives.columns if c in df_v2_extracted_controls.columns]
df_v2_full = pd.concat([
    df_p3_positives[common_cols],
    df_v2_extracted_controls[common_cols]
], ignore_index=True)

print(f'Assembled Version 2.0 Feature Table: {len(df_v2_full)} total rows ({len(df_p3_positives)} positive, {len(df_v2_extracted_controls)} controls)')
print('Class balance:', df_v2_full['label'].value_counts().to_dict())

# Save primary dataset outputs
df_v2_full.to_csv('v2_feature_table_experimental.csv', index=False)

# Save Provenance Metadata
provenance = {
    'extraction_timestamp': datetime.utcnow().isoformat(),
    'status': 'EXTRACTION_COMPLETED_COLAB',
    'gee_project': 'landslidedetection-506008',
    'target_roi': [26.0, 35.0, 74.0, 97.0],
    'candidate_control_terminology': 'Candidate Hillslope Control / Unrecorded Failure Status',
    'confound_policy': 'Confound mitigation must be empirically verified via KS tests upon ingestion; not assumed a priori.',
    'sar_screening_characterization': 'Microwave temporal consistency screening criterion (std <= 1.5 dB); not geodetic stability proof.',
    'candidate_selection_strategy': 'Minimum three_feature_topographic_distance among sampled valid candidates pooled across 3 batches',
    'twi_hydrological_resolution': 'MERIT Hydro 90m native grid preserved for upa; alpha = upa*1e6/90m; SRTM 30m slope aggregated (mean) to 90m grid',
    'total_observations': len(df_v2_full),
    'positive_events': int((df_v2_full['label'] == 1).sum()),
    'candidate_hillslope_controls': int((df_v2_full['label'] == 0).sum()),
    'slope_bounds_deg': [6.1, 41.4],
    'elevation_bounds_m': [266, 3027],
    'buffer_exclusion_m': 2000,
    'sar_stability_filter': {'max_std_db': 1.5, 'window_days': [180, 15], 'min_observations': 3},
    'land_cover_exclusions': {'esa_worldcover_snow_ice_class': 70, 'esa_worldcover_water_class': 80, 'jrc_water_max_pct': 10},
    'sampling_audit': sampling_audit,
    'positive_exclusions_audit': positive_events_excluded_audit,
    'cross_event_duplicate_controls': int(duplicate_controls_cross_event),
    'min_pairwise_control_separation_km': float(min_pairwise_ctrl_dist_km),
    'extraction_quality_audit': quality_audit,
    'matching_distribution_summary': df_dist_summary.to_dict(orient='records')
}
with open('v2_extraction_provenance.json', 'w') as f:
    json.dump(provenance, f, indent=2)

print('\nAll Version 2.0 Artifacts Exported Successfully:')
print('1. phase2_v2_hillslope_controls.csv')
print('2. v2_control_matching_diagnostics.csv')
print('3. v2_positive_events_excluded_audit.csv')
print('4. v2_matching_distribution_summary.csv')
print('5. v2_feature_table_experimental.csv')
print('6. v2_extraction_provenance.json')

### Step 10: Download Artifacts to Local Workspace
Run the cell below to download all 6 exported files to your local workstation.
Then place them into your local project directory:
- `outputs/phase2_v2_hillslope_controls.csv`
- `artifacts/susceptibility/v2_control_matching_diagnostics.csv`
- `artifacts/susceptibility/v2_positive_events_excluded_audit.csv`
- `artifacts/susceptibility/v2_matching_distribution_summary.csv`
- `artifacts/susceptibility/v2_feature_table_experimental.csv`
- `artifacts/metadata/v2_extraction_provenance.json`

In [ ]:
from google.colab import files
files.download('phase2_v2_hillslope_controls.csv')
files.download('v2_control_matching_diagnostics.csv')
files.download('v2_positive_events_excluded_audit.csv')
files.download('v2_matching_distribution_summary.csv')
files.download('v2_feature_table_experimental.csv')
files.download('v2_extraction_provenance.json')